# 🤖 Notebook 05 — Modeling & Evaluation
**Owner:** Member 5

**Goal:** Build churn prediction models, evaluate them, and identify the key factors driving churn.

**Input:** `data/merged/churn_master.csv`

**Output:** Model comparison, feature importance, and business recommendations.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

sns.set_theme(style='whitegrid')
print('All libraries imported!')

## Step 1: Load the Master Dataset

In [ ]:
master = pd.read_csv('../data/merged/churn_master.csv')
print(f'Dataset: {master.shape[0]:,} customers × {master.shape[1]} features')
print(f'Churn Rate: {master["is_churned"].mean()*100:.1f}%')
master.head()

## Step 2: Prepare Features and Target

In [ ]:
# Separate features (X) and target (y)
# Drop non-feature columns: co_ref (ID) and is_churned (target)
X = master.drop(columns=['is_churned', 'co_ref'])
y = master['is_churned']

print(f'Features: {X.shape}')
print(f'Target: {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# Check if there are any remaining categorical columns that need encoding
cat_cols = X.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    print(f'Categorical columns to encode: {list(cat_cols)}')
    # One-hot encode them
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    print(f'After encoding: {X.shape}')
else:
    print('No categorical columns — all features are numeric ✅')

## Step 3: Train/Test Split

In [ ]:
# 80% train, 20% test, stratified to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTrain churn rate: {y_train.mean()*100:.1f}%')
print(f'Test churn rate:  {y_test.mean()*100:.1f}%')

In [ ]:
# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## Step 4: Model 1 — Logistic Regression (Baseline)

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr))

## Step 5: Model 2 — Decision Tree

In [ ]:
# Train Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

# Predict
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print('=== Decision Tree ===')
print(classification_report(y_test, y_pred_dt))

## Step 6: Model 3 — Random Forest

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf))

---
## Step 7: Model Comparison Table

In [ ]:
# Compare all models
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Decision Tree': (y_pred_dt, y_prob_dt),
    'Random Forest': (y_pred_rf, y_prob_rf)
}

results = []
for name, (y_pred, y_prob) in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob)
    })

comparison = pd.DataFrame(results).set_index('Model').round(4)
print('\n📊 MODEL COMPARISON')
print('=' * 70)
comparison

## Step 8: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (y_pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/12_confusion_matrices.png', dpi=150)
plt.show()

## Step 9: ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71']
for (name, (_, y_prob)), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/13_roc_curves.png', dpi=150)
plt.show()

## Step 10: Feature Importance (Random Forest)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Get feature importances from Random Forest
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True).tail(10)

importances.plot(kind='barh', ax=ax, color='#3498db', edgecolor='black')
ax.set_title('Top 10 Features Driving Churn (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance')

plt.tight_layout()
plt.savefig('../reports/figures/14_feature_importance.png', dpi=150)
plt.show()

---
## Step 11: Conclusions & Recommendations

### TODO: Fill in after running the models

**Best Model:** ___ (based on F1/AUC)

**Top Factors Driving Churn:**
1. ...
2. ...
3. ...

**Business Recommendations:**
1. ...
2. ...
3. ...

**Limitations:**
1. ...
2. ...